<a href="https://colab.research.google.com/github/ingridmidory/Escuela-Nacional-Preparatoria-No.4-Vidal-Casta-eda-y-N-jera-/blob/main/Horarios_por_grupo_y_Salon_Copia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import zipfile
import glob
import os

In [ ]:
df = pd.read_excel("/content/Horarios 25-26.xlsx")

In [ ]:
# ==============================
# CONFIGURACIÓN
# ==============================

# Mapear días y horas
dias_map = {"LU":"Lunes","MA":"Martes","MI":"Miércoles","JU":"Jueves","VI":"Viernes"}
horas_map = {
    1:"07:00–07:50", 2:"07:50–08:40", 3:"08:40–09:30", 4:"09:30–10:20",
    5:"10:20–11:10", 6:"11:10–12:00", 7:"12:00–12:50", 8:"12:50–13:40",
    9:"13:40–14:30", 10:"14:30–15:20", 11:"15:20–16:10", 12:"16:10–17:00",
    13:"17:00–17:50", 14:"17:50–18:40", 15:"18:40–19:30", 16:"19:30–20:20",
    17:"20:20–21:10", 18:"21:10-22:00"
}
dias = list(dias_map.keys())
modulos = list(range(1,19))

dias = list(dias_map.keys())
modulos = list(range(1,19))

In [ ]:
# ==============================
# NORMALIZAR GRUPOS
# ==============================
def normalizar_grupo(grupo):
    return ''.join(filter(str.isdigit, str(grupo)))  # 401A -> 401

df['GRUPO_NORMAL'] = df['GRUPO'].apply(normalizar_grupo)

In [ ]:
def generar_pdf(tabla, nombre_archivo, titulo, fontsize=9):
    with PdfPages(nombre_archivo) as pdf:
        fig, ax = plt.subplots(figsize=(8.5, 11))
        ax.axis('off')

        # Colocar el título manualmente en coordenadas de la figura
        plt.suptitle(titulo, fontsize=14, weight='bold', y=0.98)

        # Crear la tabla
        tabla_plot = ax.table(
            cellText=tabla.values,
            rowLabels=[horas_map[h] for h in tabla.index],
            colLabels=[dias_map[d] for d in tabla.columns],
            cellLoc='center',
            loc='upper center',   # ubica la tabla en la parte superior del Axes
            edges='closed'
        )

        # Ajustes visuales
        tabla_plot.auto_set_font_size(False)
        tabla_plot.set_fontsize(fontsize)

        # Ajustar altura de filas y estilos
        for (i, j), cell in tabla_plot.get_celld().items():
            cell.set_text_props(multialignment='center', wrap=True, fontsize=fontsize)
            cell.set_width(0.4)  # ajustar según el contenido
            cell.set_height(0.2)
            if i == 0 or j == -1:
                cell.set_facecolor('#add8e6')

        pdf.savefig(fig, bbox_inches='tight')
        plt.close()

In [ ]:
# ==============================
# GENERAR HORARIOS POR GRUPO
# ==============================
os.makedirs("horarios_grupos", exist_ok=True)

for grupo, datos in df.groupby("GRUPO_NORMAL"):
    tabla = pd.DataFrame("", index=modulos, columns=dias)
    for _, row in datos.iterrows():
        dia, hora = row["HORA"][:2], int(row["HORA"][2:])
        texto = f"{row['MATERIA']}\n{row['PROFESOR(A)']}\n{row['SALON']}"
        if tabla.loc[hora, dia] != "":
            tabla.loc[hora, dia] += "\n---\n" + texto  # combinar optativas
        else:
            tabla.loc[hora, dia] = texto
    generar_pdf(tabla, f"horarios_grupos/horario_grupo_{grupo}.pdf", f"Horario Grupo {grupo}")


In [ ]:
# ==============================
# GENERAR HORARIOS POR SALÓN
# ==============================
os.makedirs("horarios_salones", exist_ok=True)

for salon, datos in df.groupby("SALON"):
    tabla = pd.DataFrame("", index=modulos, columns=dias)
    for _, row in datos.iterrows():
        dia, hora = row["HORA"][:2], int(row["HORA"][2:])
        texto = f"{row['GRUPO']}\n{row['MATERIA']}\n{row['PROFESOR(A)']}"
        if tabla.loc[hora, dia] != "":
            tabla.loc[hora, dia] += "\n---\n" + texto
        else:
            tabla.loc[hora, dia] = texto
    generar_pdf(tabla, f"horarios_salones/horario_salon_{salon}.pdf", f"Horario Salón {salon}", fontsize=10)

In [ ]:
# ==============================
# CREAR ARCHIVOS ZIP
# ==============================
with zipfile.ZipFile("horarios_grupos.zip", 'w') as zipf:
    for pdf in glob.glob("horarios_grupos/*.pdf"):
        zipf.write(pdf)

with zipfile.ZipFile("horarios_salones.zip", 'w') as zipf:
    for pdf in glob.glob("horarios_salones/*.pdf"):
        zipf.write(pdf)

print("✅ Horarios generados y comprimidos en horarios_grupos.zip y horarios_salones.zip")

✅ Horarios generados y comprimidos en horarios_grupos.zip y horarios_salones.zip
